In [0]:
# Create Spark DataFrame for Spending table
import pyspark.sql.functions as F
spending_data = [
    (1, "2019-07-01", "mobile", 100),
    (1, "2019-07-01", "desktop", 100),
    (2, "2019-07-01", "mobile", 100),
    (2, "2019-07-02", "mobile", 100),
    (3, "2019-07-01", "desktop", 100),
    (3, "2019-07-02", "desktop", 100)
]
spending_schema = ["user_id", "spend_date", "platform", "amount"]
spending_df = spark.createDataFrame(spending_data, spending_schema)

# Convert spend_date to date type using to_date()
spending_df = spending_df.withColumn("spend_date", F.to_date("spend_date"))

# Create SQL table and view
spending_df.write.mode("overwrite").saveAsTable("spending")
spark.sql("CREATE OR REPLACE VIEW spending_view AS SELECT * FROM spending")

# Display the DataFrame
display(spending_df)

In [0]:
spark.sql("""
    WITH P as (
        select distinct spend_date as spend_date, "desktop" as platform from spending
        UNION 
        select distinct spend_date as spend_date, "mobile" as platform from spending
        union
        select distinct spend_date as spend_date, "both" as platform from spending
    ), 

    UserDailyStats AS (
        SELECT 
            spend_date, 
            user_id, 
            CASE 
                WHEN COUNT(DISTINCT platform) = 2 THEN 'both' 
                ELSE MAX(platform) 
            END AS platform,
            SUM(amount) AS amount
        FROM Spending
        GROUP BY spend_date, user_id
    )

    SELECT P.spend_date, P.platform, ifnull(SUM(U.amount),0 ) as Total_Amount, COUNT(DISTINCT U.user_id) as TotalUsers from P LEFT JOIN UserDailyStats U 
    ON P.spend_date = U.spend_date AND P.platform = U.platform
    group by P.spend_date, P.platform
""").display()